<a href="https://colab.research.google.com/github/STARG-LEE/colab/blob/main/notebooks/ESG_01_Stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [개별기업] 주가 수집 & 분석

| 항목 | 내용 |
|---|---|
| **수집 방법** | 네이버 모바일 API 직접 수집 |
| **수집 항목** | 수정주가 + 원본주가 + 수정거래량 + 원본거래량 |
| **이어받기** | 마지막 저장일 이후 자동 업데이트 |
| **분석** | 캔들차트 · 이동평균 · RSI · MACD · 볼린저밴드 · 수익률 · 시뮬레이션 |
| **DB** | `○○○.○○○.○○○.○○ (비밀) / cha_stock / tbl_stock_price` |

> 전체 기업 일괄 수집은 `Batch_01_Stock.ipynb`을 사용하세요.


---
# ⚙️ 섹션 0 : 분석가 설정 (이 셀 하나만 수정하면 됩니다)

> **세 노트북(ESG_01 · ESG_02 · ESG_03)에서 아래 설정 셀은 글자 하나까지 똑같습니다.**
> 한 번만 값을 채운 뒤, 같은 내용을 세 노트북에 그대로 붙여넣으세요.
> 다른 셀은 건드릴 필요가 없습니다. 위에서 아래로 실행만 하면 됩니다.

| 무엇을 정해야 하나 | 변수 | 쓰는 곳 |
|---|---|---|
| 분석할 기업 | `STOCK_CODE` · `COMPANY_NAME` · `SEARCH_KEYWORD` · `TOPIC_CODE` | 01·02·03 공통 |
| DB 접속 | `DB_*` | 01·02·03 공통 |
| 주가 차트/시뮬레이션 | `STOCK_START_DATE` · `CHART_YEARS` · `BUY_DATE` · `INVEST_AMT` | 01 |
| 뉴스 기간·옵션 | `START_DATE` · `END_DATE` · `PHOTO_TYPE` · `REG_START_OFFSET` | 03 |
| 외부 키 | `OPENAI_API_KEY` · `GITHUB_TOKEN` (Colab 🔑 비밀) | 03 |


In [ ]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║                 ⚙️  분 석 가  설 정   (ANALYST CONFIG)              ║
# ║   이 셀은 ESG_01 · ESG_02 · ESG_03 에서 100% 동일합니다.            ║
# ║   여기 적힌 값만 바꾸면 됩니다. 아래 다른 셀은 수정하지 마세요.     ║
# ║   ⚠️ 비밀번호·API키는 코드에 적지 않습니다 → 아래 (5) 비밀 참고     ║
# ╚════════════════════════════════════════════════════════════════════╝
import os as _os

def _secret(name, default=''):
    """비밀값을 Colab 🔑(비밀) → 환경변수 순서로 읽음. 코드에 평문 저장 안 함."""
    try:
        from google.colab import userdata as _ud
        v = _ud.get(name)
        if v:
            return v
    except Exception:
        pass
    return _os.environ.get(name, default)

# ┌─ 1) 분석 대상 기업  (01·02·03 공통) ───────────────────────────────┐
STOCK_CODE     = '051900'         # 종목코드 6자리
COMPANY_NAME   = 'LG생활건강'     # 회사명 (그래프 제목 등에 표시)
SEARCH_KEYWORD = 'LG생활건강'     # 뉴스 검색어            (03 사용)
TOPIC_CODE     = 'LGHNH'          # 뉴스 테이블 접미사(영문) (03 사용)

# ┌─ 2) DB 접속  (01·02·03 공통) ──────────────────────────────────────┐
#    호스트/아이디/비밀번호는 코드에 적지 않습니다.
#    Colab 🔑(비밀)에 DB_HOST · DB_USER · DB_PASSWORD 를 등록하세요.
DB_HOST     = _secret('DB_HOST')
DB_USER     = _secret('DB_USER')
DB_PASSWORD = _secret('DB_PASSWORD')
DB_NAME     = 'cha_stock'         # DB 이름(비밀 아님)
TABLE_NAME  = 'tbl_stock_price'   # 01 주가 테이블
FIN_TABLE   = 'tbl_financial'     # 02 재무 테이블

# ┌─ 3) [ESG_01 주가] 전용 ────────────────────────────────────────────┐
STOCK_START_DATE = '19970101'     # 주가 수집 시작일(YYYYMMDD) · 이어받기 자동
CHART_YEARS = 3                   # 차트 기본 표시 기간(년)
MA_PERIODS  = [5, 20, 60, 120, 240]   # 이동평균선
BUY_DATE    = '2022-01-03'        # 투자 시뮬레이션 매수일
INVEST_AMT  = 10_000_000          # 투자 시뮬레이션 금액(원)

# ┌─ 4) [ESG_03 뉴스·ESG] 전용 ────────────────────────────────────────┐
START_DATE = '2021-01-01'         # 뉴스 수집 시작일(포함)
END_DATE   = '2026-04-08'         # 뉴스 수집 종료일(포함)
PHOTO_TYPE = 3                    # 0=전체 1=포토 2=동영상 3=지면 4=보도자료
REG_START_OFFSET = 0              # 회귀 시작연도: 0=전체 / +N=앞N년 제외 / -N=자동보정

MAX_WORKERS          = 5          # 동시 처리 스레드 수
SIMILARITY_THRESHOLD = 0.4        # ESG 이슈 매칭 코사인 임계값
ESG_TABLE            = 'ESG_ISSUES_EMBED'
TOP_N_ISSUES         = 5          # 상위 ESG 이슈 개수
MOVING_AVG_WINDOW    = 12         # 이동평균 개월
MAX_LAG_MONTHS       = 12         # 시차 회귀 최대 개월
N_CLUSTERS           = 4          # 주제 군집 수
MIN_ARTICLES         = 10
TARGET_ESG_ISSUES    = []

# ┌─ 5) 외부 API 키 / 비밀  (03에서만 필요 · 01/02는 무시) ────────────┐
#    Colab: 왼쪽 🔑(비밀) 메뉴에 아래 이름으로 등록하세요.
#      OPENAI_API_KEY · GITHUB_TOKEN · DB_HOST · DB_USER · DB_PASSWORD
OPENAI_KEY   = _secret('OPENAI_API_KEY')
GITHUB_TOKEN = _secret('GITHUB_TOKEN')

GITHUB_USERNAME = 'sdkparkforbi'  # ← 본인 GitHub 계정으로 변경 (03 HTML 업로드용)
GITHUB_REPO     = 'STOCK_ANALYZE' # 바꾸지 말 것
GITHUB_BRANCH   = 'main'          # 바꾸지 말 것
GITHUB_FOLDER   = 'storybooks'    # 바꾸지 말 것

print('⚙️ 분석가 설정 로드 완료:', COMPANY_NAME, '('+STOCK_CODE+')')
if not (DB_HOST and DB_USER and DB_PASSWORD):
    print('   ⚠️ DB 비밀 미설정 — Colab 🔑 비밀에 DB_HOST/DB_USER/DB_PASSWORD 등록 필요')


---
# 섹션 2 : 설치 및 초기화

In [2]:
!pip install -q pymysql koreanize-matplotlib mplfinance
print('설치 완료!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 6.0 MB/s eta 0:00:00
설치 완료!


In [3]:
import ast, socket, time, warnings
from datetime import datetime
import matplotlib
import matplotlib.pyplot as plt
# Colab 인라인 표시 설정
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except: pass
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import mplfinance as mpf
import numpy as np
import pandas as pd
import pymysql, requests
from sqlalchemy import create_engine, text
import koreanize_matplotlib
warnings.filterwarnings('ignore')

# DB가 없으면 먼저 생성
_conn0 = pymysql.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWORD,
                          charset='utf8mb4', autocommit=True)
_conn0.cursor().execute(
    'CREATE DATABASE IF NOT EXISTS `'+DB_NAME+'`'
    ' DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci')
_conn0.close()
print('DB 확인/생성 완료:', DB_NAME)

engine = create_engine(
    'mysql+pymysql://'+DB_USER+':'+DB_PASSWORD
    +'@'+DB_HOST+'/'+DB_NAME+'?charset=utf8mb4')

MPF_FONT = {'font.family': plt.rcParams['font.family']}
PALETTE  = {
    'up':'#EF4444','down':'#3B82F6',
    'ma5':'#FF6B6B','ma20':'#FFD93D','ma60':'#6BCB77','ma120':'#4D96FF','ma240':'#C77DFF',
    'blue':'#3B82F6','green':'#10B981','red':'#EF4444',
    'amber':'#F59E0B','purple':'#8B5CF6','teal':'#14B8A6','pink':'#EC4899','gray':'#6B7280',
}

print('초기화 완료!')
print('  분석 기업:', COMPANY_NAME, '('+STOCK_CODE+')')
print('  DB:', DB_HOST, '/', DB_NAME, '/', TABLE_NAME)


OperationalError: (2003, "Can't connect to MySQL server on 'localhost' ([Errno 111] Connection refused)")

---
# 섹션 3 : DB 테이블 준비

In [ ]:
with engine.begin() as conn:
    conn.execute(text(
        'CREATE TABLE IF NOT EXISTS '+TABLE_NAME+' ('
        '  id         INT AUTO_INCREMENT PRIMARY KEY,'
        '  date       DATE NOT NULL,'
        '  open       INT, high INT, low INT,'
        '  close      INT,'
        '  close_raw  INT,'
        '  diff       INT,'
        '  volume     BIGINT,'
        '  volume_raw BIGINT,'
        '  code       VARCHAR(10) NOT NULL,'
        '  company    VARCHAR(100),'
        '  UNIQUE KEY uq (code, date)'
        ') ENGINE=InnoDB DEFAULT CHARSET=utf8mb4'))

    # 기존 컬럼 없으면 자동 추가 (마이그레이션)
    for col, defn in [('close_raw','INT DEFAULT NULL AFTER `close`'),
                       ('volume_raw','BIGINT DEFAULT NULL AFTER `volume`')]:
        chk = conn.execute(text(
            'SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS'
            ' WHERE TABLE_SCHEMA=:db AND TABLE_NAME=:tb AND COLUMN_NAME=:col'),
            {'db':DB_NAME,'tb':TABLE_NAME,'col':col}).fetchone()[0]
        if chk == 0:
            conn.execute(text('ALTER TABLE '+TABLE_NAME+' ADD COLUMN '+col+' '+defn))
            print('  컬럼 추가:', col)

print('DB 테이블 준비 완료:', TABLE_NAME)


---
# 섹션 4 : 수집 함수 정의

네이버 모바일 API에서 **수정주가**와 **원본주가**를 모두 가져와
**수정거래량**을 계산합니다.

```
수정비율   = 원본종가 ÷ 수정종가
수정거래량 = 원본거래량 × 수정비율
```


In [ ]:
def _fetch_naver(code, start_time, end_time, request_type=1):
    """네이버 API 단일 호출 (request_type: 1=수정, 0=원본)"""
    for attempt in range(3):
        try:
            url = (
                'https://m.stock.naver.com/front-api/external/chart/domestic/info?'
                'symbol='+code+'&requestType='+str(request_type)
                +'&startTime='+start_time+'&endTime='+end_time+'&timeframe=day'
            )
            resp = requests.get(url, timeout=10)
            if resp.status_code != 200: time.sleep(1); continue
            data_list = ast.literal_eval(resp.text.replace('\n','').replace(' ',''))
            if len(data_list) <= 1: return None
            df = pd.DataFrame(data_list[1:], columns=data_list[0])
            df = df.rename(columns={'날짜':'date','시가':'open','고가':'high',
                                     '저가':'low','종가':'close','거래량':'volume'})
            keep = [c for c in ['date','open','high','low','close','volume'] if c in df.columns]
            df = df[keep].dropna()
            if df.empty: return None
            df['date'] = pd.to_datetime(df['date'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
            for col in ['open','high','low','close','volume']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            df = df[(df['open']>0)&(df['high']>0)&(df['low']>0)&(df['close']>0)]
            return df.sort_values('date').reset_index(drop=True)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError, socket.timeout):
            time.sleep(2*(attempt+1))
        except Exception: return None
    return None

def fetch_stock(code, start_time, end_time, full_mode=False):
    """
    full_mode=True  → 수정(requestType=1) + 원본(requestType=0) 모두 수집 → 수정거래량 계산
    full_mode=False → 수정만 수집 (일별 이어받기용)
    """
    df_adj = _fetch_naver(code, start_time, end_time, request_type=1)
    if df_adj is None: return None

    if full_mode:
        df_raw = _fetch_naver(code, start_time, end_time, request_type=0)
        if df_raw is not None and not df_raw.empty:
            merged = pd.merge(
                df_adj.rename(columns={'volume':'vol_adj_tmp'}),
                df_raw[['date','close','volume']].rename(
                    columns={'close':'close_raw','volume':'volume_raw'}),
                on='date', how='inner')
            adj_ratio = merged['close_raw'] / merged['close'].replace(0, np.nan)
            merged['volume'] = (merged['volume_raw']*adj_ratio
                               ).fillna(merged['volume_raw']).round().astype(int)
            merged['diff'] = merged['close'].diff().fillna(0).astype(int)
            return merged[['date','open','high','low','close','close_raw',
                            'diff','volume','volume_raw']].sort_values('date').reset_index(drop=True)
        # 원본 수집 실패 시 차선책
        df_adj['close_raw']  = df_adj['close']
        df_adj['volume_raw'] = df_adj['volume']
    else:
        df_adj['close_raw']  = df_adj['close']
        df_adj['volume_raw'] = df_adj['volume']

    df_adj['diff'] = df_adj['close'].diff().fillna(0).astype(int)
    return df_adj[['date','open','high','low','close','close_raw',
                   'diff','volume','volume_raw']].sort_values('date').reset_index(drop=True)

print('수집 함수 준비 완료!')


---
# 섹션 5 : 주가 수집 실행

```
DB에 데이터 없음  → 전체 기간 수집 (full_mode, 수정거래량 계산)
DB에 데이터 있음  → 마지막 저장일 이후만 추가 (이어받기)
분할·병합 감지    → 종가 1% 이상 차이 → 전체 재수집
```


In [ ]:
from tqdm.auto import tqdm

today_str = datetime.now().strftime('%Y%m%d')

# ── 현재 DB 상태 확인 ──────────────────────────────────────────
with engine.connect() as conn:
    row = conn.execute(text(
        'SELECT date, close FROM '+TABLE_NAME
        +' WHERE code=:c ORDER BY date DESC LIMIT 1'),
        {'c': STOCK_CODE}).fetchone()

if row is None:
    # ── 최초 수집 ─────────────────────────────────────────────
    print('DB에 데이터 없음 → 전체 기간 수집 시작 ('+STOCK_START_DATE+' ~ '+today_str+')')
    df_new = fetch_stock(STOCK_CODE, STOCK_START_DATE, today_str, full_mode=True)
    mode = 'full'

else:
    last_date_obj  = row[0]   # datetime.date
    last_close_db  = int(row[1])
    last_date_str  = last_date_obj.strftime('%Y-%m-%d')
    last_date_ymd  = last_date_obj.strftime('%Y%m%d')
    print('기존 데이터 마지막 날짜:', last_date_str, '/ 종가:', f'{last_close_db:,}원')

    # 마지막 날 ~ 오늘 구간 수집
    df_check = fetch_stock(STOCK_CODE, last_date_ymd, today_str, full_mode=False)

    if df_check is None or df_check.empty:
        print('이미 최신 상태입니다.')
        df_new = None; mode = 'skip'
    else:
        # 분할·병합 감지
        overlap = df_check[df_check['date'] == last_date_str]
        if not overlap.empty:
            api_close  = int(overlap['close'].iloc[0])
            diff_ratio = abs(api_close - last_close_db) / max(last_close_db, 1)
            if diff_ratio > 0.01:
                print(f'분할·병합 감지! DB종가={last_close_db:,} / API종가={api_close:,} → 전체 재수집')
                with engine.begin() as conn:
                    conn.execute(text('DELETE FROM '+TABLE_NAME+' WHERE code=:c'),{'c':STOCK_CODE})
                df_new = fetch_stock(STOCK_CODE, STOCK_START_DATE, today_str, full_mode=True)
                mode = 'resplit'
            else:
                # 정상 이어받기 — 새 날짜만
                df_new = df_check[df_check['date'] > last_date_str].reset_index(drop=True)
                mode = 'incremental'
        else:
            df_new = df_check[df_check['date'] > last_date_str].reset_index(drop=True)
            mode = 'incremental'

# ── DB 저장 ────────────────────────────────────────────────────
if df_new is not None and not df_new.empty:
    with engine.begin() as conn:
        conn.execute(text(
            'INSERT IGNORE INTO '+TABLE_NAME
            +' (date,open,high,low,close,close_raw,diff,volume,volume_raw,code,company)'
            ' VALUES (:da,:op,:hi,:lo,:cl,:cr,:di,:vo,:vr,:co,:cm)'),
            [dict(da=r['date'],op=int(r['open']),hi=int(r['high']),lo=int(r['low']),
                  cl=int(r['close']),cr=int(r['close_raw']),di=int(r['diff']),
                  vo=int(r['volume']),vr=int(r['volume_raw']),
                  co=STOCK_CODE,cm=COMPANY_NAME)
             for _,r in df_new.iterrows()])

    mode_label = {'full':'최초 전체 수집','incremental':'이어받기','resplit':'분할 재수집'}.get(mode,mode)
    print(f'저장 완료! [{mode_label}]  {len(df_new):,}행 추가')
elif mode != 'skip':
    print('수집된 데이터 없음 — 종목코드를 확인하세요:', STOCK_CODE)


---
# 섹션 6 : 데이터 로드 & 기술 지표 계산

In [ ]:
df = pd.read_sql(
    'SELECT * FROM '+TABLE_NAME+' WHERE code='+repr(STOCK_CODE)+' ORDER BY date',
    engine)

if len(df) == 0:
    print('데이터가 없습니다. 섹션 5를 먼저 실행하세요.')
else:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    # 이동평균선
    for p in MA_PERIODS:
        df[f'MA{p}'] = df['close'].rolling(p).mean()

    # RSI (14일)
    delta = df['close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    # 볼린저 밴드 (20일, 2σ)
    df['BB_mid']   = df['close'].rolling(20).mean()
    df['BB_std']   = df['close'].rolling(20).std()
    df['BB_upper'] = df['BB_mid'] + 2*df['BB_std']
    df['BB_lower'] = df['BB_mid'] - 2*df['BB_std']

    # MACD (12, 26, 9)
    ema12 = df['close'].ewm(span=12).mean()
    ema26 = df['close'].ewm(span=26).mean()
    df['MACD']        = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

    # 수익률 / 변동성 / 낙폭
    df['returns']  = df['close'].pct_change()
    df['roll_vol'] = df['returns'].rolling(20).std() * np.sqrt(252) * 100
    df['drawdown'] = (df['close'] / df['close'].cummax() - 1) * 100

    d252 = df.tail(252)
    print('데이터 로드 완료!')
    print(f'  총 거래일  : {len(df):,}일')
    print(f'  기간       : {df["date"].min().strftime("%Y-%m-%d")} ~ {df["date"].max().strftime("%Y-%m-%d")}')
    print(f'  최근 종가  : {int(df["close"].iloc[-1]):,}원')
    print(f'  52주 고가  : {int(d252["close"].max()):,}원')
    print(f'  52주 저가  : {int(d252["close"].min()):,}원')
    print(f'  RSI (14)   : {df["RSI"].iloc[-1]:.1f}')


---
# 섹션 7 : 종합 현황 대시보드

핵심 지표와 최근 주가 흐름을 한눈에 확인합니다.

In [ ]:
latest = df.iloc[-1]; prev = df.iloc[-2]
diff_v = int(latest['close']) - int(prev['close'])
diff_p = diff_v / int(prev['close']) * 100
ytd_df = df[df['date'].dt.year == datetime.now().year]
ytd_ret = (int(latest['close']) / int(ytd_df['close'].iloc[0]) - 1)*100 if len(ytd_df)>0 else 0
vol_avg = int(df['volume'].tail(20).mean())

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)
fig.patch.set_facecolor('#F0F4F8')

# 핵심 지표 카드
ax0 = fig.add_subplot(gs[0, :])
ax0.set_facecolor('#F0F4F8'); ax0.axis('off')
metrics = [
    ('현재가',          f'{int(latest["close"]):,}원',        PALETTE['blue']),
    ('전일비',          ('+' if diff_v>=0 else '')+f'{diff_v:,}원\n({diff_p:+.2f}%)',
                        PALETTE['red'] if diff_v>=0 else PALETTE['down']),
    ('52주 고가',       f'{int(d252["close"].max()):,}원',     PALETTE['green']),
    ('52주 저가',       f'{int(d252["close"].min()):,}원',     PALETTE['amber']),
    ('YTD 수익률',      f'{ytd_ret:+.2f}%',                    PALETTE['green'] if ytd_ret>=0 else PALETTE['red']),
    ('20일 평균거래량', f'{vol_avg:,}주',                      PALETTE['purple']),
    ('RSI (14)',        f'{df["RSI"].iloc[-1]:.1f}',           PALETTE['amber']),
]
for i, (label, val, color) in enumerate(metrics):
    x = i/7 + 0.01
    ax0.add_patch(mpatches.FancyBboxPatch((x,0.05), 0.12, 0.9,
        boxstyle='round,pad=0.02', facecolor='white',
        edgecolor=color, linewidth=2, transform=ax0.transAxes))
    ax0.text(x+0.06, 0.72, label, ha='center', va='center', fontsize=10,
             color='#6B7280', transform=ax0.transAxes)
    ax0.text(x+0.06, 0.35, val,   ha='center', va='center', fontsize=13,
             color=color, transform=ax0.transAxes, fontweight='bold')
ax0.set_title(COMPANY_NAME+' ('+STOCK_CODE+') 주가 현황',
              fontsize=16, fontweight='bold', pad=10)

# 주가 추이 (최근 CHART_YEARS년)
cutoff = df['date'].max() - pd.DateOffset(years=CHART_YEARS)
df_c   = df[df['date'] >= cutoff].copy()
ax1 = fig.add_subplot(gs[1, :])
ax1.fill_between(df_c['date'], df_c['close'], alpha=0.06, color=PALETTE['blue'])
ax1.plot(df_c['date'], df_c['close'], color=PALETTE['blue'], linewidth=1.5, label='종가(수정)')
for p, c in zip([20, 60, 120], ['#FFD93D','#6BCB77','#4D96FF']):
    col = f'MA{p}'
    valid = df_c[df_c[col].notna()]
    if len(valid) > 0:
        ax1.plot(valid['date'], valid[col], color=c, linewidth=1.2, alpha=0.85, label=f'MA{p}')
ax1.fill_between(df_c['date'], df_c['BB_lower'], df_c['BB_upper'],
                alpha=0.06, color=PALETTE['purple'], label='볼린저밴드')
ax1.set_title(f'주가 추이 (최근 {CHART_YEARS}년)',fontsize=12,fontweight='bold')
ax1.legend(loc='upper left', fontsize=9, ncol=6)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'{x:,.0f}'))
ax1.grid(True, alpha=0.2)

# 수정거래량
ax2 = fig.add_subplot(gs[2, :2])
clrs_v = [PALETTE['up'] if c >= o else PALETTE['down']
          for c, o in zip(df_c['close'], df_c['open'])]
ax2.bar(df_c['date'], df_c['volume']/1e4, color=clrs_v, alpha=0.7, width=1)
ax2.plot(df_c['date'], df_c['volume'].rolling(20).mean()/1e4,
         color='black', linewidth=1.5, label='20일 평균')
ax2.set_title('수정거래량 (만주)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.2, axis='y')

# RSI
ax3 = fig.add_subplot(gs[2, 2])
ax3.plot(df_c['date'], df_c['RSI'], color=PALETTE['purple'], linewidth=1.5)
ax3.axhline(70, color=PALETTE['red'],   linestyle='--', alpha=0.7, label='과매수(70)')
ax3.axhline(30, color=PALETTE['green'], linestyle='--', alpha=0.7, label='과매도(30)')
ax3.fill_between(df_c['date'], 70, df_c['RSI'], where=df_c['RSI']>=70,
                alpha=0.15, color=PALETTE['red'])
ax3.fill_between(df_c['date'], 30, df_c['RSI'], where=df_c['RSI']<=30,
                alpha=0.15, color=PALETTE['green'])
ax3.set_ylim(0, 100); ax3.set_title('RSI (14)', fontsize=11, fontweight='bold')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.2)

# MACD
ax4 = fig.add_subplot(gs[2, 3])
ax4.plot(df_c['date'], df_c['MACD'],        color=PALETTE['blue'], linewidth=1.5, label='MACD')
ax4.plot(df_c['date'], df_c['MACD_signal'], color=PALETTE['red'],  linewidth=1.2, label='Signal')
clrs_m = [PALETTE['up'] if v>=0 else PALETTE['down'] for v in df_c['MACD_hist']]
ax4.bar(df_c['date'], df_c['MACD_hist'], color=clrs_m, alpha=0.5, width=1)
ax4.axhline(0, color='gray', linewidth=0.8)
ax4.set_title('MACD', fontsize=11, fontweight='bold')
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.2)

plt.suptitle(COMPANY_NAME+' 주가 종합 대시보드', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('company_stock_dashboard.png', dpi=120, bbox_inches='tight'); plt.show()
print('대시보드 완료!')


---
# 섹션 8 : 캔들스틱 차트 (전문가 뷰)

In [ ]:
CANDLE_MONTHS = 6   # ← 원하는 개월 수

cutoff    = df['date'].max() - pd.DateOffset(months=CANDLE_MONTHS)
df_candle = df[df['date'] >= cutoff].copy().set_index('date')
ohlcv     = df_candle[['open','high','low','close','volume']].copy()

ap_list = []
for p, c, w in zip([5,20,60,120,240],
                   [PALETTE['ma5'],PALETTE['ma20'],PALETTE['ma60'],PALETTE['ma120'],PALETTE['ma240']],
                   [0.8, 1.2, 1.5, 1.5, 2.0]):
    col = f'MA{p}'
    if col in df_candle.columns and df_candle[col].notna().sum() > 5:
        ap_list.append(mpf.make_addplot(df_candle[col], color=c, width=w, panel=0))
if df_candle['BB_upper'].notna().sum() > 5:
    ap_list.append(mpf.make_addplot(df_candle['BB_upper'], color='#C77DFF', width=0.8, linestyle='--', panel=0))
    ap_list.append(mpf.make_addplot(df_candle['BB_lower'], color='#C77DFF', width=0.8, linestyle='--', panel=0))

mc    = mpf.make_marketcolors(up=PALETTE['up'], down=PALETTE['down'],
                               edge='inherit', wick='inherit', volume='inherit')
style = mpf.make_mpf_style(marketcolors=mc, gridstyle=':', gridcolor='#E5E7EB',
                            facecolor='white', figcolor='white', rc=MPF_FONT)
dr = (df_candle.index.min().strftime('%Y.%m.%d')
      + ' ~ ' + df_candle.index.max().strftime('%Y.%m.%d'))
mpf.plot(ohlcv, type='candle', style=style,
         title='\n'+COMPANY_NAME+'  캔들스틱  |  '+dr,
         ylabel='주가(원)', ylabel_lower='수정거래량',
         volume=True, addplot=ap_list, figsize=(18, 10),
         datetime_format='%Y-%m', show_nontrading=False)
print('캔들스틱 차트 완료!')


---
# 섹션 9 : 수익률 & 리스크 분석

In [ ]:
periods = {'1개월':21,'3개월':63,'6개월':126,'1년':252,'3년':756,'5년':1260}
ret_data = {lbl: (df['close'].iloc[-1]/df['close'].iloc[-days]-1)*100
            for lbl,days in periods.items() if len(df)>days}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(COMPANY_NAME+' 수익률 & 리스크 분석', fontsize=16, fontweight='bold')

# 누적 수익률
ax = axes[0,0]
cutoff = df['date'].max() - pd.DateOffset(years=CHART_YEARS)
df_r   = df[df['date'] >= cutoff].copy()
cum    = (1 + df_r['returns'].fillna(0)).cumprod() - 1
ax.plot(df_r['date'], cum*100, color=PALETTE['blue'], linewidth=2)
ax.fill_between(df_r['date'], 0, cum*100,
                where=cum>=0, alpha=0.15, color=PALETTE['green'])
ax.fill_between(df_r['date'], 0, cum*100,
                where=cum<0,  alpha=0.15, color=PALETTE['red'])
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title(f'누적 수익률 (최근 {CHART_YEARS}년)', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'{x:+.1f}%'))
ax.grid(True, alpha=0.2)

# 기간별 수익률
ax = axes[0,1]
clrs = [PALETTE['green'] if v>=0 else PALETTE['red'] for v in ret_data.values()]
bars = ax.bar(list(ret_data.keys()), list(ret_data.values()), color=clrs, width=0.5)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_title('기간별 수익률', fontsize=12, fontweight='bold')
for bar, v in zip(bars, ret_data.values()):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height() + (0.5 if v>=0 else -2),
            f'{v:+.1f}%', ha='center', fontsize=10, fontweight='bold',
            color=PALETTE['green'] if v>=0 else PALETTE['red'])
ax.grid(True, alpha=0.2, axis='y')

# 롤링 변동성
ax = axes[1,0]
ax.plot(df_r['date'], df_r['roll_vol'], color=PALETTE['amber'], linewidth=1.5)
ax.fill_between(df_r['date'], 0, df_r['roll_vol'], alpha=0.2, color=PALETTE['amber'])
ax.set_title('20일 롤링 변동성 (연율화, %)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.2)

# MDD
ax = axes[1,1]
ax.fill_between(df_r['date'], 0, df_r['drawdown'], alpha=0.5, color=PALETTE['red'])
ax.plot(df_r['date'], df_r['drawdown'], color='#991B1B', linewidth=1)
mdd = df_r['drawdown'].min()
ax.axhline(mdd, color='black', linestyle='--', linewidth=1.2, label=f'MDD: {mdd:.1f}%')
ax.set_title('최대낙폭 (Drawdown)', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('company_returns.png', dpi=120, bbox_inches='tight'); plt.show()

ann_ret = df['returns'].mean()*252*100
ann_vol = df['returns'].std()*np.sqrt(252)*100
sharpe  = ann_ret/ann_vol if ann_vol>0 else 0
print('수익률 분석 요약')
print(f'  연평균 수익률: {ann_ret:+.2f}%')
print(f'  연간 변동성  : {ann_vol:.2f}%')
print(f'  샤프비율     : {sharpe:.2f}')
print(f'  최대낙폭(MDD): {df["drawdown"].min():.2f}%')


---
# 섹션 10 : 투자 시뮬레이션

> *"그때 샀더라면 지금 얼마나 벌었을까?"* 💭

섹션 1의 `BUY_DATE`와 `INVEST_AMT`를 변경해보세요.


In [ ]:
df_after = df[df['date'] >= pd.to_datetime(BUY_DATE)]
if len(df_after) == 0:
    print('해당 날짜 이후 데이터가 없습니다.')
else:
    actual_buy = df_after.iloc[0]['date']
    buy_price  = int(df_after.iloc[0]['close'])
    cur_price  = int(df.iloc[-1]['close'])
    cur_date   = df.iloc[-1]['date']
    shares     = INVEST_AMT // buy_price
    invested   = shares * buy_price
    cur_value  = shares * cur_price
    profit     = cur_value - invested
    total_ret  = (cur_price/buy_price - 1)*100
    hold_days  = (pd.to_datetime(cur_date) - pd.to_datetime(actual_buy)).days
    hold_years = hold_days / 365.25
    cagr = ((cur_price/buy_price)**(1/hold_years)-1)*100 if hold_years>0.01 else total_ret

    df_sim = df[df['date'] >= pd.to_datetime(actual_buy)].copy()
    df_sim['자산가치'] = shares * df_sim['close']
    df_sim['수익률']   = (df_sim['close']/buy_price - 1)*100

    fig, axes = plt.subplots(3, 1, figsize=(18, 14),
                              gridspec_kw={'height_ratios':[3,2,1]})
    fig.suptitle(f'{COMPANY_NAME} 투자 시뮬레이션',
                 fontsize=16, fontweight='bold')

    ax = axes[0]
    ax.plot(df_sim['date'], df_sim['close'], color=PALETTE['blue'], linewidth=2)
    ax.axhline(buy_price, color=PALETTE['red'],   linestyle='--', linewidth=1.8,
              label=f'매수가 {buy_price:,}원')
    ax.axhline(cur_price, color=PALETTE['green'], linestyle='-.', linewidth=1.8,
              label=f'현재가 {cur_price:,}원')
    ax.fill_between(df_sim['date'], buy_price, df_sim['close'],
                   where=df_sim['close']>=buy_price, alpha=0.12,
                   color=PALETTE['green'], label='수익 구간')
    ax.fill_between(df_sim['date'], buy_price, df_sim['close'],
                   where=df_sim['close']<buy_price, alpha=0.12,
                   color=PALETTE['red'], label='손실 구간')
    ax.scatter([actual_buy], [buy_price], color=PALETTE['red'],
              s=200, zorder=5, marker='^', label=f'매수일 {str(actual_buy)[:10]}')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'{x:,.0f}'))
    ax.set_title('주가 추이', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left'); ax.grid(True, alpha=0.2)

    ax = axes[1]; axt = ax.twinx()
    ax.fill_between(df_sim['date'], invested, df_sim['자산가치'],
                   where=df_sim['자산가치']>=invested, alpha=0.25, color=PALETTE['green'])
    ax.fill_between(df_sim['date'], invested, df_sim['자산가치'],
                   where=df_sim['자산가치']<invested,  alpha=0.25, color=PALETTE['red'])
    ax.plot(df_sim['date'], df_sim['자산가치'],
            color=PALETTE['green'] if profit>=0 else PALETTE['red'], linewidth=2)
    ax.axhline(invested, color='gray', linestyle='--', linewidth=1)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'{x:,.0f}'))
    ax.set_title('자산 가치 변화', fontsize=12, fontweight='bold')
    axt.plot(df_sim['date'], df_sim['수익률'], color=PALETTE['amber'],
            linewidth=1.2, alpha=0.7, linestyle=':')
    axt.axhline(0, color='gray', linewidth=0.5)
    axt.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'{x:+.1f}%'))
    axt.set_ylabel('수익률 (%)', color=PALETTE['amber'])
    ax.grid(True, alpha=0.2)

    ax = axes[2]
    clrs_v = [PALETTE['up'] if c>=o else PALETTE['down']
              for c,o in zip(df_sim['close'], df_sim['open'])]
    ax.bar(df_sim['date'], df_sim['volume']/1e4, color=clrs_v, alpha=0.7, width=1)
    ax.set_title('수정거래량', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2, axis='y')

    plt.tight_layout()
    plt.savefig('company_simulation.png', dpi=120, bbox_inches='tight'); plt.show()

    emoji = '🎉' if profit>0 else '😢'
    print(emoji, '투자 시뮬레이션 결과')
    print('='*45)
    print(f'  매수일    : {str(actual_buy)[:10]}  (보유 {hold_days:,}일)')
    if str(actual_buy)[:10] != BUY_DATE:
        print(f'  (요청일 {BUY_DATE} → 다음 거래일로 이동)')
    print(f'  매수가    : {buy_price:>12,}원')
    print(f'  현재가    : {cur_price:>12,}원')
    print(f'  매수 주수 : {shares:>12,}주')
    print(f'  투자금액  : {invested:>12,}원')
    print(f'  현재가치  : {cur_value:>12,}원')
    print(f'  수익금액  : {("+" if profit>=0 else "")}{profit:>11,}원')
    print(f'  총수익률  : {total_ret:>+10.2f}%')
    print(f'  CAGR      : {cagr:>+10.2f}%')
    print('='*45)
    if   cagr > 20: print(f'  연 {cagr:.1f}% — 워런 버핏 수준!')
    elif cagr > 10: print(f'  연 {cagr:.1f}% — 코스피 장기 평균 상회!')
    elif cagr > 0:  print(f'  연 {cagr:.1f}% — 플러스 수익')
    else:           print(f'  연 {cagr:.1f}% — 손실 구간')
